# crunchy — MCNP parsing in Python

[`crunchy`](https://github.com/AlvaroCubi/crunchy) is a fast, **lossless** MCNP
input parser written in Rust, with Python bindings. This notebook tours its
capabilities on a small self-contained model:

1. Parse and inspect the model (cells, surfaces, materials, transforms).
2. Prove the parse is **byte-for-byte lossless**.
3. **Renumber the whole geometry** — definitions *and* every reference.
4. **Edit per card** — values (`cell.material`, `cell.density`) and geometry
   (add/remove surfaces and `#n` complements).
5. **Build & remove whole cards** programmatically, and `validate()` the result.

In [1]:
import crunchy
print("crunchy version:", crunchy.__version__)

## The input model

A small pin-cell-ish model exercising comments, an inline `$` comment, a `&` continuation, a union (`:`), a cell complement (`#4`), a `LIKE n BUT` cell, an `RPP` macrobody, and a transformed surface.

In [2]:
MODEL = 'Crunchy demo: a small pin-cell-ish model\nc --- cells ---\n1 1 -10.5 -1 imp:n=1                  $ fuel pin\n2 2 -1.0  1 -2 imp:n=1                $ water gap\n3 0 (2 -3) #4 imp:n=1                 $ moderator minus the insert\n4 3 -2.7 -40 imp:n=1 imp:p=1 &\n     vol=8.0                          $ aluminium insert (macrobody)\n5 0 3 : 20 imp:n=0                    $ graveyard\n10 like 1 but mat=4 rho=-11.3         $ second pin, different fuel\n\nc --- surfaces ---\n1 CZ 0.5\n2 CZ 0.6\n3 CZ 5.0\n20 SO 100\n30 1 PZ 50                            $ surface positioned by transform 1\n40 RPP -1 1 -1 1 -1 2\n\nc --- data ---\nm1 92235.80c 0.04 92238.80c 0.96      $ enriched uranium\nm2 1001.31c 2 8016.31c 1              $ light water\nmt2 lwtr.10t\nm3 13027.80c 1                        $ aluminium\nm4 92235.80c 0.20 92238.80c 0.80\ntr1 0 0 5\nmode n\nsdef pos=0 0 0 erg=2.0\nf4:n 1 2 4\n'
print(MODEL)

In [3]:
model = crunchy.parse(MODEL)          # or crunchy.Model.from_file("model.mcnp")
model

## Parse summary

The parser is recoverable — a clean parse yields no diagnostics.

In [4]:
print("cells:      ", model.num_cells)
print("surfaces:   ", model.num_surfaces)
print("materials:  ", len(model.materials))
print("transforms: ", len(model.transforms))
print("diagnostics:", model.diagnostics)

## Surfaces (typed)

Each surface exposes its number, mnemonic, coefficients, and an optional transform.

In [5]:
for s in model.surfaces:
    tr = f"  (TR{s.transform})" if s.transform else ""
    print(f"{s.id:>3}  {s.kind:<4} {s.coeffs}{tr}")

## Cells & geometry references

Cells expose material/density, void status, the surfaces they reference (with sense), cell complements, and the `LIKE n` base.

In [6]:
for c in model.cells:
    if c.like is not None:
        print(f"{c.id:>3}  LIKE {c.like} BUT ...")
        continue
    kind = "void" if c.is_void else f"mat {c.material} @ {c.density}"
    extra = f"  complements #{c.cell_refs}" if c.cell_refs else ""
    print(f"{c.id:>3}  {kind:<14} surfaces {c.signed_surfaces}{extra}")

## Materials & transforms

In [7]:
for m in model.materials:
    comp = "  ".join(f"{z}={frac}" for z, frac in m.entries)
    print(f"m{m.id:<2} {comp}")

print()
for t in model.transforms:
    print(f"tr{t.id}  displacement {t.displacement}  degrees={t.degrees}")

## Fast id lookups

`model.surface(id)` / `model.cell(id)` use a cached index — no need to scan the whole list.

In [ ]:
print(model.surface(40))
print(model.cell(3))
print(model.material(1))

## Lossless round-trip

Re-emitting the model reproduces the input **byte-for-byte** — comments, spacing, and continuations included.

In [ ]:
assert str(model) == MODEL
print("byte-for-byte lossless:", str(model) == MODEL)

## Whole-geometry renumbering

The headline capability: renumbering updates **definitions and every reference**
consistently — signed surfaces in geometry, `#n` complements, and `LIKE n`
bases. A mapping may be a callable or a `dict`.


In [ ]:
def cell_line(text, cell_id):
    for line in text.splitlines():
        toks = line.split()
        if toks and toks[0] == str(cell_id):
            return line
    return None

print("cell 3 before:", cell_line(str(model), 3))

model.offset_surfaces(1000)            # every surface +1000 (defs + refs)
model.renumber_cells(lambda n: n + 900)  # every cell   +900  (defs + #n + LIKE)

print("cell 3 after: ", cell_line(str(model), 903))

Notice inside cell 903: surfaces `2 -3` became `1002 -1003`, the complement `#4` became `#904`, and the cell id itself `3` became `903`. Everything else on the line — the `$` comment and spacing — is untouched.

In [ ]:
edited = str(model)
print("comment preserved:      ", "$ moderator minus the insert" in edited)
print("continuation preserved: ", "imp:p=1 &" in edited)
print("surface def renumbered:  ", "1040 RPP -1 1 -1 1 -1 2" in edited)
print("LIKE base renumbered:    ", "910 like 901 but" in edited)

In [ ]:
print(edited)

## Per-card editing & exploration

The typed objects are **live handles** on the model. Explore by `cell.text` — the
exact card source, inline `$` comments included — then assign `cell.material` /
`cell.density` to edit in place. The change writes straight through the lossless
engine, so only those numbers move.

In [ ]:
demo = crunchy.parse(MODEL)

# Find a cell by its text, then edit material & density in place.
for cell in demo.cells:
    if "$ fuel pin" in cell.text:
        cell.material = 5
        cell.density = -10.9

print(cell_line(str(demo), 1))

## Structural geometry editing

Restructure a cell's region: add or remove surfaces and `#n` complements. Only
the edited card is re-emitted (its inline comment and parameters preserved);
every other card stays byte-for-byte.

In [ ]:
c = demo.cell(2)
assert c
print("before:", c.signed_surfaces)
c.add_surface(-3)      # intersect the region with -3
c.add_complement(4)    # AND in  #4
c.remove_surface(1)    # drop surface 1 (either sense)
print("after: ", c.signed_surfaces, " complements", c.cell_refs)
print(cell_line(str(demo), 2))

## Building & removing cards

Add whole cards from MCNP text — each returns a live handle you can keep
editing. `remove_cell` / `remove_surface` delete a card by number, and
`validate()` reports any dangling references.

In [ ]:
demo2 = crunchy.parse(MODEL)

demo2.add_surface("50 SO 200")
demo2.add_material("m9 6000 1")
new = demo2.add_cell("20 9 -2.0 -50")   # live handle to the new cell
new.add_surface(1)                       # ...editable like any other cell

print(cell_line(str(demo2), 20))
print("materials:", demo2.num_materials, " surfaces:", demo2.num_surfaces)
print("validate:", demo2.validate())     # [] — model is consistent

demo2.remove_surface(50)                  # break a reference on purpose
print("after removing surface 50:", demo2.validate())

## Save

```python
model.save("model_renumbered.mcnp")
```

That's the full loop: **parse → inspect → edit → emit**, losslessly, from Python.